# LinearRegression from Scratch


# 구현할 것
- 공부시간과 성적간의 관계를 모델링한다.
    - **머신러닝 모델(모형)이란** 수집한 데이터를 기반으로 입력값(Feature)와 출력값(Target)간의 관계를 하나의 공식으로 정의한 함수이다. 그 공식을 찾는 과정을 **모델링**이라고 한다.
    - 이 예제에서는 공부한 시험시간으로 점수를 예측하는 모델을 정의한다.
    - 입력값과 출력값 간의 관계를 정의할 수있는 다양한 함수(공식)이 있다. 여기에서는 딥러닝과 관계가 있는 **Linear Regression** 을 사용해본다.

# 데이터 확인
- 입력데이터: 공부시간
- 출력데이터: 성적

|공부시간|점수|
|-|-|
|1|20|
|2|40|
|3|60|

우리가 수집한 공부시간과 점수 데이터를 바탕으로 둘 간의 관계를 식으로 정의 할 수 있으면 **내가 몇시간 공부하면 점수를 얼마 받을 수 있는지 예측할 수 있게 된다.**   
수집한 데이터를 기반으로 앞으로 예측할 수있는 모형을 만드는 것이 머신러닝 모델링이다.

  

## 학습(훈련) 데이터셋 만들기
- 모델을 학습시키기 위한 데이터셋을 구성한다.
- 입력데이터와 출력데이터을 각각 다른 행렬로 구성한다.
- 하나의 데이터 포인트의 입력/출력 값은 같은 index에 정의한다.

### 선형회귀 (Linear Regression)
- Feature들의 가중합을 이용해 Target을 추정한다.
- Feature에 곱해지는 가중치(weight)들은 각 Feature가 Target 얼마나 영향을 주는지 영향도가 된다.
    - 음수일 경우는 target값을 줄이고 양수일 경우는 target값을 늘린다.
    - 가중치가 0에 가까울 수록 target에 영향을 주지 않는 feature이고 0에서 멀수록 target에 많은 영향을 준다.
- 모델 학습과정에서 가장 적절한 Feature의 가중치를 찾아야 한다.
      

\begin{align}
&\large \hat{y} = W\cdot X + b\\
&\small \hat{y}: \text{모델추정값}\\
&\small W: \text{가중치}\\
&\small X: \text{Feature(입력값)}\\
&\small b: \text{bias(편향)}
\end{align}



## Train dataset 구성
- Train data는 feature(input)와 target(output) 각각 2개의 행렬로 구성한다.
- Feature의 행은 관측치(개별 데이터)를 열을 Feature(특성, 변수)를 표현한다. 이 문제에서는 `공부시간` 1개의 변수를 가진다.
- Target은 모델이 예측할 대상으로 행은 개별 관측치, 열은 각 항목에 대한 정답으로 구성한다.   
  이 문제에서 예측할 항목은 `시험점수` 한개이다.

In [1]:
import torch

X_train = torch.tensor([[1.0], [2.0], [3.0]])
y_train = torch.tensor([[20.0], [40.0], [60.0]])

X_train.size(), y_train.size()

(torch.Size([3, 1]), torch.Size([3, 1]))

## 파라미터 (weight, bias) 정의
- 학습대상/최적화 대상

In [ ]:
### 파라미터 선정
# weight: 형태 - feature가 1개(1, 출력값의 개수-1)
weight = torch.randn(1, 1, requires_grad=True) # 최적화 대상. shape - [1: feature 수, 1: 정답 개수]
bias = torch.randn(1, requires_grad=True) # 1: 정답 개수

print(weight.size(), bias.size())

torch.Size([1, 1]) torch.Size([1])


In [3]:
weight

tensor([[1.8864]], requires_grad=True)

In [4]:
bias

tensor([0.2551], requires_grad=True)

In [ ]:
### 추론
pred = X_train @ weight + bias
print("추론한 시험 점수:", pred)

추론한 시험 점수: tensor([[2.1415],
        [4.0278],
        [5.9142]], grad_fn=<AddBackward0>)


In [ ]:
### 오차 계산(MSE)
loss = torch.mean((pred - y_train)**2, dim=0) # pred는 weight, bias를 이용해서 연산했기 때문에 gradient를 계산해야 하는 값이 됨 -> grad_fn이 저장됨
loss

tensor([1512.7321], grad_fn=<MeanBackward1>)

In [ ]:
### weight와 bias 업데이트
# 1. gradient 계산(loss / weight, loss / bias)
loss.backward() # 변화율을 보고자하는 최종 결과에 backward 적용


In [9]:
print(weight.data, bias.data)
print(weight.grad, bias.grad)

tensor([[1.8864]]) tensor([0.2551])
tensor([[-168.0402]]) tensor([-71.9443])


In [10]:
# 2. weight, bias 업데이트 : 현재값 - 학습률 * grad
lr = 0.1
weight.data = weight.data - lr * weight.grad
bias.data = bias.data - lr * bias.grad

print("새 weight, bias")
print(weight.data, bias.data)

새 weight, bias
tensor([[18.6904]]) tensor([7.4495])


In [11]:
# 다시 추론
pred2 = X_train @ weight + bias
print(pred2)

tensor([[26.1399],
        [44.8303],
        [63.5207]], grad_fn=<AddBackward0>)


In [12]:
# 오차
torch.mean((pred2 - y_train)**2)

tensor(24.4752, grad_fn=<MeanBackward0>)

In [ ]:
loss # loss가 0이 될 때까지 반복

tensor([1512.7321], grad_fn=<MeanBackward1>)

### 모델링

In [15]:
# 모델 정의(함수 or 클래스로 정의)

weight = torch.randn(1, 1, requires_grad=True)
bias = torch.randn(1, requires_grad=True)

def linear_model(X):
    return X @ weight + bias

In [17]:
# 오차 계산 함수(MSE)
def mse_loss(pred, y):
    return torch.mean((pred-y)**2)

### 학습
1. 모델을 이용해 추정한다.
   - pred = model(input)
1. loss를 계산한다.
   - loss = loss_fn(pred, target)
1. 계산된 loss를 파라미터에 대해 미분하여 계산한 gradient 값을 각 파라미터에 저장한다.
   - loss.backward()
1. optimizer를 이용해 파라미터를 update한다.
   - optimizer.step()  
1. 파라미터의 gradient(미분값)을 0으로 초기화한다.
   - optimizer.zero_grad()
- 위의 단계를 반복한다.   

In [ ]:
epochs = 2000 # 최적화 작업을 몇 번 반복할지(train set을 몇 번 학습할 건지)
lr = 0.01

for epoch in range(epochs):
    # 1. 추론
    pred = linear_model(X_train)
    # 2. 오차 계산
    loss = mse_loss(pred, y_train)
    # 3. 파라미터들에 대한 gradient 계산(backward 하는 이유 : 파라미터 업데이트)
    loss.backward()
    # 4. 파라미터 업데이트(최적화)
    weight.data = weight.data - lr * weight.grad
    bias.data = bias.data - lr * bias.grad
    # 5. 파라미터 gradient 초기화
    weight.grad = None
    bias.grad = None

    if epoch % 100 == 0 or epoch == (epochs-1): # 100번 반복당, 마지막에 loss 출력 (로그)
        print(f"[{epoch+1}/{epochs}] - Loss : {loss.item()}")

[1/2000] - Loss : 1876.9635009765625
[101/2000] - Loss : 5.300662994384766
[201/2000] - Loss : 3.2754924297332764
[301/2000] - Loss : 2.024052143096924
[401/2000] - Loss : 1.2507437467575073
[501/2000] - Loss : 0.7728831171989441
[601/2000] - Loss : 0.47759440541267395
[701/2000] - Loss : 0.29512324929237366
[801/2000] - Loss : 0.1823679655790329
[901/2000] - Loss : 0.11269208788871765
[1001/2000] - Loss : 0.06963804364204407
[1101/2000] - Loss : 0.04303175210952759
[1201/2000] - Loss : 0.026590684428811073
[1301/2000] - Loss : 0.016431177034974098
[1401/2000] - Loss : 0.010153483599424362
[1501/2000] - Loss : 0.006274320650845766
[1601/2000] - Loss : 0.0038769065868109465
[1701/2000] - Loss : 0.002395843854174018
[1801/2000] - Loss : 0.0014804474776610732
[1901/2000] - Loss : 0.0009148407843895257
[2000/2000] - Loss : 0.0005681180045939982


In [20]:
weight.data, bias.data

(tensor([[19.9724]]), tensor([0.0628]))

In [22]:
with torch.no_grad():
    p = linear_model(X_train)
    print(p)

tensor([[20.0352],
        [40.0075],
        [59.9799]])


In [23]:
time = torch.tensor([[6], [8.2]], dtype=torch.float32)
with torch.no_grad():
    p2 = linear_model(time)
    print(p2)

tensor([[119.8971],
        [163.8363]])


# 다중 입력, 다중 출력
- 다중입력: Feature가 여러개인 경우
- 다중출력: Output 결과가 여러개인 경우

다음 가상 데이터를 이용해 사과와 오렌지 수확량을 예측하는 선형회귀 모델을 정의한다.  
[참조](https://www.kaggle.com/code/aakashns/pytorch-basics-linear-regression-from-scratch)


|온도(F)|강수량(mm)|습도(%)|사과생산량(ton)|오렌지생산량|
|-|-|-|-:|-:|
|73|67|43|56|70|
|91|88|64|81|101|
|87|134|58|119|133|
|102|43|37|22|37|
|69|96|70|103|119|

```
사과수확량  = w11 * 온도 + w12 * 강수량 + w13 * 습도 + b1
오렌지수확량 = w21 * 온도 + w22 * 강수량 + w23 *습도 + b2
```

- `온도`, `강수량`, `습도` 값이 **사과**와, **오렌지 수확량**에 어느정도 영향을 주는지 가중치를 찾는다.
    - 모델은 사과의 수확량, 오렌지의 수확량 **두개의 예측결과를 출력**해야 한다.
    - 사과에 대해 예측하기 위한 weight 3개와 오렌지에 대해 예측하기 위한 weight 3개 이렇게 두 묶음, 총 6개의 weight를 정의하고 학습을 통해 가장 적당한 값을 찾는다.
        - `개별 과일를 예측하기 위한 weight들 @ feature들` 의 계산 결과를  **Node, Unit, Neuron** 이라고 한다.
        - 두 과일에 대한 Unit들을 묶어서 **Layer** 라고 한다.
- 목적은 우리가 수집한 train 데이터셋을 이용해 **정확한 예측을 위한 weight와 bias 들**을 찾는 것이다.

## Train Dataset
- Train data는 feature(input)와 target(output) 각각 2개의 행렬로 구성한다.
- Feature의 행은 관측치(개별 데이터)를 열을 Feature(특성, 변수)를 표현한다. 이 문제에서는 `온도, 강수량, 습도` 세개의 변수를 가진다.
- Target은 모델이 예측할 대상으로 행은 개별 관측치, 열은 각 항목에 대한 정답으로 구성한다. 이 문제에서 예측할 항목은 `사과수확량, 오렌지 수확량` 2개의 값이다.

In [ ]:
#  input: 생산환경 (temp, rainfall, humidity) : (5, 3)
environs = [
    [73, 67, 43], 
    [91, 88, 64], 
    [87, 134, 58], 
    [102, 43, 37], 
    [69, 96, 70]
]

# Targets: 생산량 - (apples, oranges) - (5, 2)
apple_orange_output = [
    [56, 70], 
    [81, 101], 
    [119, 133], 
    [22, 37], 
    [103, 119]
]

In [ ]:
import torch
# Dataset을 torch.Tensor로 생성
X = torch.tensor(environs, dtype=torch.float32)
y = torch.tensor(apple_orange_output, dtype=torch.float32)
X.shape, y.shape

In [ ]:
X

## weight와 bias
- weight: 각 feature들이 생산량에 영향을 주었는지의 가중치로 feature에 곱해줄 값.
    - 사과, 오렌지의 생산량을 구해야 하므로 가중치가 두개가 된다.
    - weight의 shape: `(3, 2)`
- bias는 모든 feature들이 0일때 생산량이 얼마일지를 나타내는 값으로 feature와 weight간의 가중합 결과에 더해줄 값이다.
    - 사과, 오렌지의 생산량을 구하므로 bias가 두개가 된다.
    - bias의 shape: `(2, )`

### Linear Regression model
모델은 weights `w`와 inputs `x`의 내적(dot product)한 값에 bias `b`를 더하는 함수.

$$
\hspace{2.5cm} X \hspace{1.1cm} \cdot \hspace{1.2cm} W \hspace{1.2cm}  + \hspace{1cm} b \hspace{2cm}
$$

$$
\left[ \begin{array}{cc}
73 & 67 & 43 \\
91 & 88 & 64 \\
\vdots & \vdots & \vdots \\
69 & 96 & 70
\end{array} \right]
%
\cdot
%
\left[ \begin{array}{cc}
w_{11} & w_{21} \\
w_{12} & w_{22} \\
w_{13} & w_{23}
\end{array} \right]
%
+
%
\left[ \begin{array}{cc}
b_{1} & b_{2} \\
b_{1} & b_{2} \\
\vdots & \vdots \\
b_{1} & b_{2} \\
\end{array} \right]
$$


<center style="font-size:0.9em">
$w_{11},\,w_{12},\,w_{13}$: 사과 생산량 계산시 각 feature들(생산환경)에 곱할 가중치   <br>
$w_{21},\,w_{22},\,w_{23}$: 오렌지 생산량 계산시 각 feature들(생산환경)에 곱할 가중치    
</center>

<center>
<img src="https://raw.githubusercontent.com/kgmyhGit/image_resource/main/deeplearning/figures/3_unit_layer.png">
</center>

##  모델링

# pytorch built-in 모델을 사용해 Linear Regression 구현

In [ ]:
inputs = torch.tensor(
    [[73, 67, 43], 
     [91, 88, 64], 
     [87, 134, 58], 
     [102, 43, 37], 
     [69, 96, 70]], dtype=torch.float32)

In [ ]:
targets = torch.tensor(
    [[56, 70], 
    [81, 101], 
    [119, 133], 
    [22, 37], 
    [103, 119]], dtype=torch.float32)

## torch.nn.Linear
Pytorch는 torch.nn.Linear 클래스를 통해 Linear Regression 모델을 제공한다.  
torch.nn.Linear에 입력 feature의 개수와 출력 값의 개수를 지정하면 random 값으로 초기화한 weight와 bias들을 생성해 모델을 구성한다.
- `torch.nn.Linear(input feature의 개수 , output 값의 개수)`

## Optimizer와 Loss 함수 정의
- **Optimizer**: 계산된 gradient값을 이용해 파라미터들을 업데이트 하는 함수
- **Loss 함수**: 정답과 모델이 예측한 값사이의 차이(오차)를 계산하는 함수.
  - 모델을 최적화하는 것은 이 함수의 값을 최소화하는 것을 말한다. 
- `torch.optim` 모듈에 다양한 Optimizer 클래스가 구현되있다.
- `torch.nn` 또는 `torch.nn.functional` 모듈에 다양한 Loss 함수가 제공된다. 

In [ ]:
# 선형회귀 모델을 정의. torch.nn.Linear 클래스
import torch
import torch.nn as nn

model = nn.Linear(3, 2)  # 3: input feature 개수, 2: output 수

In [ ]:
# loss 함수
loss_fn = torch.nn.MSELoss()  # 클래스
# loss_fn = torch.nn.functional.mse_loss # 함수 

In [ ]:
# optimizer (torch.optim 모듈에 정의): weight.data = weight.data - lr * weight.grad
optimizer = torch.optim.SGD(
    model.parameters(), # 최적화 대상 파라미터들을 model에서 조회해서 전달.
    lr = 0.00001,       # Learning Rage
)

In [ ]:
list(model.parameters())

## Model Train

In [ ]:
epochs = 5000

for epoch in range(epochs):
    # 추론
    pred = model(inputs)  
    # loss 계산
    loss = loss_fn(pred, targets) # torch.nn.functional.mse_loss(pred, targets) # (모델추정값, 정답)
    # gradient 계산
    loss.backward()
    # 파라미터 업데이트: optimizer.step()
    optimizer.step()
    # 파라미터 초기화 w.grad=None, b.grad=None
    optimizer.zero_grad()
    # 현재 epoch 학습 결과를 log로 출력
    if epoch % 100 == 0 or epoch == epochs-1:
        print(f"[{epoch+1:04d}/{epochs}] - {loss.item()}")

In [ ]:
# 추론 => gradient 계산을 할 필요가 없다. ==> grad_fn을 만들 필요가 없다. 그래서 torch.no_grad() 블록에서 추론 작업을 실행한다.
with torch.no_grad():
    pred = model(inputs)

In [ ]:
targets

In [ ]:
pred

In [ ]:
# 학습 로직을 함수 구현
def train(inputs, targets, epochs, model, loss_fn, optimizer):

    for epoch in range(epochs):
        # 추론
        pred = model(inputs)
        # loss 계산
        loss = loss_fn(pred, targets) # torch.nn.functional.mse_loss(pred, targets) # (모델추정값, 정답)
        # gradient 계산
        loss.backward()
        # 파라미터 업데이트: optimizer.step()
        optimizer.step()
        # 파라미터 초기화 w.grad=None, b.grad=None
        optimizer.zero_grad()
        # 현재 epoch 학습 결과를 log로 출력
        if epoch % 100 == 0 or epoch == epochs-1:
            print(f"[{epoch+1:04d}/{epochs}] - {loss.item()}")

In [ ]:
model = nn.Linear(3, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

In [ ]:
train(inputs, targets, 5000, model, nn.functional.mse_loss, optimizer)